# L5a: Single Asset Geometric Brownian Motion and NPV
Previously, introduced geometric Brownian motion (GBM) and estimated its parameters from historical data. Today, we continue that discussion and compare GBM simulations with out-of-sample prices observed in 2025. Next, we'll compute the net present value (NPV) trade-rule for GBM forecasted prices. Finally, we'll examine the performance of the exponential moving average (EMA) updates to the GBM model parameters to potentially improve the model forecasts.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * **Interpret out-of-sample GBM predictions:** Explain how historical mean growth and volatility determine the distribution of later prices, and interpret simulated paths and pointwise prediction bands against observed prices.
> * **Calculate an NPV target probability:** Express a long position's discounted fractional return in terms of its sale price and derive the probability of exceeding a chosen target at the scheduled sale time using a future price distribution computed from the GBM model.
> * **Update GBM parameter estimates:** A criticism of the binomial model and GBM is the assumption of constant parameters. Let's examine how exponential moving averages can be used to update these parameters over time. Apply exponentially weighted updates for mean growth and volatility, explain how the half-life controls their response to new observations, and use the estimates in a forecast with fixed parameters.

Let's get started!

___

## Examples
Three examples develop the price comparison, the NPV calculation, and parameter updating:

> [▶ Implement an out-of-sample (OOS) single asset prediction](CHEME-5660-L5a-Example-OOS-SAGBM-Fall-2026.ipynb). How well does a model fitted to past data describe future prices? We use mean growth and volatility estimated from 2014 to 2024 data to simulate prices for 2025 and compare them with the observations. We check how often observed prices fall within the model's prediction bands, first for one firm and then across the dataset.

> [▶ Evaluate an NPV-based GBM trade rule](CHEME-5660-L5a-Example-GBM-NPV-TradeRule-Fall-2026.ipynb). How likely is a stock trade to exceed our target scaled NPV at a scheduled (fixed) sale time? We use the estimated GBM parameters, a holding period, and a benchmark rate to calculate this probability. We check the calculation using a target with a known probability and plot how the probability changes as we raise the target.

> [▶ Update GBM parameters with an exponential moving average](CHEME-5660-L5a-Example-EMA-SAGBM-Fall-2026.ipynb). Can giving recent observations more weight improve the forecasts from a GBM model? We initialize mean growth and volatility from the 2014–2024 estimates, update them as 2025 prices arrive, and compare their forecasts with the frozen model. We measure whether updating volatility alone or both parameters improves the predicted NPV target probabilities.

Optional extensions are collected near the end of the lecture.
___

## Company Profile: Renaissance Technologies

[Renaissance Technologies (RenTec)](https://www.rentec.com/Home.action?about=true) is a quantitative investment management firm founded by [Jim Simons (1938–2024)](https://www.simonsfoundation.org/2024/05/10/simons-foundation-co-founder-mathematician-and-investor-jim-simons-dies-at-86/), a mathematician and former chair of the [mathematics department at Stony Brook University](https://math.stonybrook.edu). Simons brought mathematical research into investment management and later supported mathematics and basic science through the [Simons Foundation](https://www.simonsfoundation.org), which he cofounded with his wife, [Marilyn Simons](https://www.simonsfoundation.org/people/marilyn-simons-2/).

Last time, we introduced Jane Street through its trading and market-making business. Renaissance gives us another connection between mathematics and finance: using statistical models to manage investment funds.

> __What makes RenTec distinctive?__
>
> * __Investment approach:__ Renaissance uses [mathematical and statistical methods](https://www.rentec.com/Home.action?index=true) to design and execute its investment programs. Historical data, models, and the software that implements them are central to its approach.
> * __Research culture:__ The firm's [research staff](https://www.rentec.com/Home.action?about=true) includes people trained in mathematics, physics, computer science, and related fields. Its [research scientist role](https://www.rentec.com/Careers.action?jobs=true&selectedPosition=researchScientist) connects data science, statistics, applied mathematics, and programming with the development of trading algorithms.
> * __Medallion Fund:__ Launched in 1988, Medallion became the firm's best-known fund for its exceptional trading performance. Using the returns reported in Gregory Zuckerman's [*The Man Who Solved the Market*](https://www.penguinrandomhouse.com/books/557104/the-man-who-solved-the-market-by-gregory-zuckerman/), [Bradford Cornell's study](https://www.cornell-capital.com/uploads/papers/medallion-fund.pdf) gives _average annual returns_ of approximately 66% before fees and 39% after fees over 1988–2018. Medallion is closed to outside investors; Renaissance's funds for outside investors use different strategies. Its record should therefore not be attributed to every Renaissance fund.

__Explore further:__

* __Jobs and internships:__ Browse [RenTec's official careers page](https://www.rentec.com/Careers.action?jobs=true) for research, programming, and systems roles. Students interested in internships should check that page for openings; no internship position is listed there as of September 18, 2026 (but keep checking!). The Simons Foundation also offers paid summer research internships in mathematics and science through its [Summer at Simons](https://www.simonsfoundation.org/summer-at-simons/) program at the [Flatiron Institute](https://www.simonsfoundation.org/flatiron/) in New York City.
* __YouTube interviews and channels:__ Watch [Jim Simons's full-length Numberphile interview](https://www.youtube.com/watch?v=QNznD9hMEh0) on the [Numberphile2 channel](https://www.youtube.com/@numberphile2), or [The mathematician who cracked Wall Street](https://www.youtube.com/watch?v=U5kIdtMJGc8) on the [TED channel](https://www.youtube.com/@TED). These conversations explore his path from mathematics to investing and philanthropy.

__Connection to today's lecture:__ Renaissance builds statistical models of prices and tests them against data. Today we do the same at classroom scale with GBM: we fit the model to 2014–2024 prices, forecast 2025, and compare the forecast with what actually happened. Let's begin by recalling the price model and the meaning of its parameters.

___

## Concept Review: Single Asset GBM Model
Recall that single asset geometric Brownian motion (GBM) describes share price changes through drift and random fluctuations, both proportional to the current share price $S(t)$ (units: USD/share). The model is given by:

$$
\begin{align*}
\frac{dS\left(t\right)}{S(t)} &= \mu\,{dt}+\sigma\,{dW(t)}.
\end{align*}
$$

Here, $S(t)$ is the current share price in USD/share, with initial price $S(0)=S_0>0$. We measure time in years, so the drift parameter $\mu\in\mathbb{R}$ has units of inverse years and the __volatility__ $\sigma>0$ parameter has units of inverse square-root years. The term $dW(t)$ is an increment of a Wiener (random noise) process:

> __Parameters__
>
> * __Drift versus growth:__ The __mean growth rate__ is $\mu_g=\mu-\sigma^{2}/2$ (units: inverse years). It is the expected value of the one-step growth rate $g_j=(1/\Delta t)\ln(S_{t_j}/S_{t_{j-1}})$, where $\Delta t>0$ is the observation interval in years. Equivalently, $\mu=\mu_g+\sigma^2/2$: the drift and mean growth rate have the same units but describe different quantities.
> * __Risk:__ We estimate the volatility parameter from the standard deviation of the observed growth rates, $\sigma_g$, adjusted for the time step: $\hat{\sigma}=\sigma_g\sqrt{\Delta t}$. With $\Delta t$ measured in years, $\sigma_g$ has units of $\mathrm{year}^{-1}$ and $\hat{\sigma}$ has units of $\mathrm{year}^{-1/2}$.
> * __Constant parameters:__ We assume $\mu$ and $\sigma$ remain fixed over the modeled period. The out-of-sample example below examines how well a model fitted to historical data describes later observations. However, constant parameters are in general __probably not a good assumption__ (but we do it anyway).

For a single fixed horizon $T>0$, we can instead sample the terminal price directly:

$$
S_T=S_0\exp\!\left[\mu_gT+\sigma\sqrt{T}\,Z\right],
\qquad Z\sim\mathcal{N}(0,1).
$$
However, we can also generate a price path by advancing one step at a time. On a grid $t_j=j\Delta t$, the price advances one step at a time by the exact __one-step transition__ from L4b:

$$
\boxed{
\begin{aligned}
S_{t_j} &= S_{t_{j-1}}\exp\!\left[\mu_g\Delta t+\sigma\sqrt{\Delta t}\,Z_j\right],
\qquad j=1,2,\ldots,N.
\end{aligned}
}
$$

The random variables $Z_1,Z_2,\ldots,Z_N$ (shocks) are independent standard normal draws, one for each time step. 
The one-step form generates a price path, while the fixed-horizon form gives the price distribution at the selected time $T$.

In the previous lecture, we estimated the mean growth rate and volatility from historical data, giving $\hat{\mu}_g$ and $\hat{\sigma}$. Here, the hats indicate estimates from data. These two estimates are all we need to simulate prices with the one-step transition. Why, then, calculate $\hat{\mu}$? The original GBM differential equation uses the drift $\mu$. To express our fitted model using that parameter, we calculate:

$$
\boxed{\hat{\mu}=\hat{\mu}_g+\frac{\hat{\sigma}^2}{2}.}
$$

If we substitute this drift estimate into the one-step solution, the half-variance terms cancel:

$$
\hat{\mu}-\frac{\hat{\sigma}^2}{2}
=\left(\hat{\mu}_g+\frac{\hat{\sigma}^2}{2}\right)
-\frac{\hat{\sigma}^2}{2}
=\hat{\mu}_g.
$$

Both forms give the same simulated prices for the same draws of $Z_j$. The conversion simply lets us express the model using either the drift $\mu$ or the mean growth rate $\mu_g$. Let's now use the estimates to describe prices observed after the fitting period.

> __Example:__
>
> [▶ Implement an out-of-sample (OOS) single asset prediction](CHEME-5660-L5a-Example-OOS-SAGBM-Fall-2026.ipynb). We use the 2014–2024 estimates to simulate prices for 2025, compare pointwise prediction bands with observed prices, and examine coverage for one asset and across the shared dataset.

The same price distribution also lets us calculate the probability of exceeding a target return at a scheduled sale time. Let's explore that idea next.

___

## GBM Trade Rule

Suppose we want to calculate the probability that the scaled NPV that we've been working with, i.e., the discounted fractional return, exceeds a target value at a scheduled (future) sale time.

> __Scenario:__ Suppose we purchase $n_0>0$ shares of ticker `XYZ` at time $0$
> for $S_0>0$ USD/share. We sell all shares at time $T=N\Delta t$ for $S_T$
> USD/share, where $N\geq1$ and $\Delta t>0$ is measured in years. We assume
> no dividends, transaction fees, or bid–ask spread.

We model the long position as an abstract asset with two cash-flow events: the purchase at entry and the sale at exit. This connects the trade to our earlier discounted cash-flow calculations.

The purchase is a cash outflow, and the sale is a cash inflow. Let $g_y$ denote
the constant, continuously compounded benchmark growth rate, measured in inverse
years. Discounting the sale proceeds to time zero gives given $g_{y}$ gives:

$$
\operatorname{NPV}(g_y,T)
=\underbrace{-n_0S_0}_{\text{purchase today}}
+\underbrace{n_0S_Te^{-g_yT}}_{\text{present value of sale proceeds}}.
$$

Dividing by the initial investment gives the __scaled NPV__:

$$
\boxed{
\rho_T
=\frac{\operatorname{NPV}(g_y,T)}{n_0S_0}
=\left(\frac{S_T}{S_0}\right)e^{-g_yT}-1.
}
$$

This dimensionless quantity $\rho_T>-1$ measures the discounted fractional return on our
initial investment. There are three cases to consider:
* __Positive scaled NPV__: A positive scaled NPV means that the present value of the
sale proceeds exceeds the purchase cost, given the benchmark growth rate $g_y$. 
* __Negative scaled NPV__: A negative scaled NPV means that the present value of the sale proceeds is less than the present value of the purchase cost, given the benchmark growth rate $g_y$. 
* __Zero scaled NPV__: A scaled NPV of zero means that the present value of the sale proceeds equals the present value of the purchase cost, given the benchmark growth rate $g_y$. Another way to think about the zero case is that we make exactly the benchmark growth rate $g_y$ on our investment.

In lecture L4a, we used a binomial lattice to describe the uncertain future sale price $S_T$. Let's now
substitute the GBM solution for $S_T$ into this expression:

$$
\begin{aligned}
\rho_T
&=\exp\!\left[\mu_gT+\sigma\sqrt T\,Z\right]e^{-g_yT}-1\\
&=\exp\!\left[\underbrace{(\mu_g-g_y)}_{\text{interesting!}}\;T+\sigma\sqrt T\,Z\right]-1,
\qquad Z\sim\mathcal N(0,1).
\end{aligned}
$$

Discounting replaces $\mu_g$ with $(\mu_g-g_y)$ in the growth term, ie., an __excess growth rate__ relative to the benchmark, while the uncertainty comes from the normally distributed shock $Z$.

> __Distribution of the scaled NPV:__ The quantity $1+\rho_T$ is lognormally
> distributed, so:
>
> $$
> \ln(1+\rho_T)
> \sim\mathcal N\!\left((\mu_g-g_y)T,\;\sigma^2T\right).
> $$
>
> The scaled NPV is then $\rho_T=(1+\rho_T)-1$, a lognormal random variable minus one.
> A lognormal random variable is always positive, so $1+\rho_T>0$ and therefore $\rho_T>-1$:
> the most we can lose is the purchase price $n_0S_0$ we pay today.

For short holding periods with $|g_y|T\ll1$, the discount factor is
approximately one, giving $\rho_T\approx S_T/S_0-1$. We will retain the exact
discounted expression so our calculation also applies to longer holding periods.

Let's use this distribution to calculate the probability that $\rho_T$
exceeds a specified target.


### Terminal Target Probability

Choose a target scaled NPV $\rho_\star>-1$. We want the probability that
$\rho_T>\rho_\star$ at the scheduled sale time $T$. Adding one to both sides and taking the logarithm preserves the inequality:

$$
\rho_T>\rho_\star
\quad\Longleftrightarrow\quad
\ln(1+\rho_T)>\ln(1+\rho_\star).
$$

Substituting our expression for the log of $1+\rho_T$ gives:

$$
(\mu_g-g_y)T+\sigma\sqrt T\,Z>\ln(1+\rho_\star).
$$

Subtracting the mean $(\mu_g-g_y)T$ and dividing by the positive standard
deviation $\sigma\sqrt T$ expresses our target condition in terms of the
standard normal random variable $Z$. The trade exceeds its target scaled NPV
at the scheduled sale time when:

$$
Z>z_\star,
\qquad
z_\star=
\frac{\ln(1+\rho_\star)-(\mu_g-g_y)T}
{\sigma\sqrt T}.
$$

> __Terminal target probability:__ For $T>0$, $\sigma>0$, and $\rho_\star>-1$,
> the probability of exceeding the target is:
>
> $$
> \boxed{
> \mathbb P(\rho_T>\rho_\star)
> =\mathbb P(Z>z_\star)
> =1-\Phi(z_\star).
> }
> $$
>
> Here, $\Phi$ is the standard normal cumulative distribution function. The
> value $\Phi(z_\star)$ is the probability that $Z\leq z_\star$; subtracting
> it from one gives the probability that $Z>z_\star$.

For targets $\rho_\star\leq-1$, the probability is one because the scaled NPV
is always greater than $-1$ under this model. These probabilities concern the
scheduled sale time; selling when a boundary is first reached requires a
separate calculation.

### Connection to the Benchmark
We can also express the target condition as a requirement on the sale price:

$$
\rho_T>\rho_\star
\quad\Longleftrightarrow\quad
S_T>\underbrace{S_0(1+\rho_\star)e^{g_yT}}_{K\,:\,\text{target sale price}}.
$$

With $\rho_\star=0$, the trade succeeds when its sale proceeds beat the chosen benchmark. Increasing $g_y$ raises the required sale price $K$ and lowers the probability of exceeding it. We can therefore examine the same trade against different assumed benchmark growth rates without changing the GBM price model.

The benchmark is an assumed constant growth rate, as in L4b; it is not another risky asset's realized price path. Both the benchmark and the target can be changed in the NPV example.

Let's evaluate the target probability using parameters estimated from historical data.

> __Example:__
>
> [▶ Evaluate an NPV-based GBM trade rule](CHEME-5660-L5a-Example-GBM-NPV-TradeRule-Fall-2026.ipynb). We calculate the probability of exceeding a target scaled NPV, verify the calculation at the median, and examine how the probability changes as we raise the target.

The median check gives a known result against which to verify the calculation. The target sweep then shows how requiring a higher discounted return lowers its probability.

___

## Updating GBM Parameters with an Exponential Moving Average
An __exponential moving average (EMA)__ gives recent observations more weight when estimating mean growth and volatility. The following result summarizes the updates used in our example.

> __Proposition: Exponentially Weighted GBM Estimates__
>
> Let $g_k=\ln(S_k/S_{k-1})/\Delta t$ be the observed growth rate ($\mathrm{year}^{-1}$), with positive prices spaced $\Delta t>0$ years apart. At entry row $s$, initialize its mean and variance from the training estimates: $m_s=\hat\mu_{g,0}$ and $v_s=\hat\sigma_0^2/\Delta t$.
>
> For a decay factor $0<\lambda<1$ and each new observation $k>s$, the weighted moments and corresponding GBM estimates satisfy:
>
> $$
> \begin{aligned}
> \delta_k&=g_k-m_{k-1},\\
> m_k&=m_{k-1}+(1-\lambda)\delta_k,\\
> v_k&=\lambda\left[v_{k-1}+(1-\lambda)\delta_k^2\right],\\[4pt]
> \hat\mu_{g,k}&=m_k,\qquad
> \hat\sigma_k=\sqrt{v_k\Delta t},\qquad
> \hat\mu_k=\hat\mu_{g,k}+\frac12\hat\sigma_k^2.
> \end{aligned}
> $$
>
> Here, $m_k$ and $v_k$ are the weighted mean ($\mathrm{year}^{-1}$) and variance ($\mathrm{year}^{-2}$) of growth rates; $\hat\mu_{g,k}$, $\hat\sigma_k$, and $\hat\mu_k$ are mean growth, volatility, and arithmetic drift. The variance accounts for the changing mean and has no finite-sample unbiased correction.

Choose $\lambda=2^{-1/H_{\mathrm{half}}}$, where $H_{\mathrm{half}}$ is the number of observations over which an older weight halves. A 21-observation half-life gives $\lambda\approx0.9675$: the updated mean assigns about 96.75% weight to the previous mean and 3.25% to the newest growth-rate observation. A shorter half-life makes the estimates respond faster but also makes them more sensitive to noise; the half-life is separate from the forecast holding period.

[▶ Derivation of the EMA parameter updates](advanced/ema-derivation/CHEME-5660-L5a-Derivation-EMA-SAGBM-Fall-2026.ipynb). Why does the variance update contain an extra factor of $\lambda$? We derive the exponential weights, centered variance, and conversion to GBM parameters.

Let's apply these updates to the 2025 observations and compare their forecasts with the frozen model.

> __Example:__
>
> [▶ Update GBM parameters with an exponential moving average](CHEME-5660-L5a-Example-EMA-SAGBM-Fall-2026.ipynb). We compare frozen parameters, updated volatility, and updated mean growth with volatility using the same observations, sale dates, and NPV targets. We evaluate their predicted probabilities against the later outcomes.

The advanced examples below examine parameter uncertainty and simulation error in more detail.

___


## Optional Advanced Material
The following L4b examples provide further detail on parameter uncertainty and simulation. They are optional and are not prerequisites for L5b.

* [▶ Uncertainty in mean growth](../../week-4/L4b/advanced/drift-uncertainty/CHEME-5660-L4b-Advanced-DriftUncertainty-Fall-2026.ipynb). Why can mean growth remain uncertain even with thousands of price observations? We derive the standard error of the fitted mean growth rate, accounting for the correlation and increasing variance of the regression errors. We compare how the observation period and sampling frequency affect the precision of mean growth and volatility estimates, then examine how uncertainty in those estimates changes the probability of exceeding a target scaled NPV.

* [▶ Monte Carlo versus the closed form](../../week-4/L4b/advanced/monte-carlo/CHEME-5660-L4b-Advanced-MonteCarlo-TargetProbability-Fall-2026.ipynb). How many simulated price paths do we need to estimate a target probability accurately? We compare simulation estimates and their standard errors with the analytical result. We also compare the exact GBM transition with a numerical approximation and test paired paths with opposite shocks, called antithetic variates, to reduce sampling error.

These examples distinguish uncertainty in the fitted parameters from error introduced by a finite number of simulated paths.

___

## Summary
In this lecture, we connected the single-asset GBM model with a long position's discounted payoff, derived its target probability, and developed parameter updates for forecasts made as new prices arrive.

> __Key Takeaways:__
>
> * **Out-of-sample price predictions:** We used the single-asset GBM transition to describe possible future prices from historical mean-growth and volatility estimates. The out-of-sample example compared its pointwise prediction bands with prices observed outside the fitting period.
> * **NPV target probabilities:** We expressed the trade as purchase and sale cash flows, scaled its NPV by the initial investment, and derived the probability of exceeding a target at the scheduled sale time. The benchmark growth rate allowed us to compare the same trade with different growth requirements.
> * **Exponential parameter updates:** We used exponentially weighted moments of growth rates to estimate mean growth, volatility, and arithmetic drift. The half-life controls how quickly older observations lose influence. Each forecast holds its estimates fixed, and the EMA example checks whether updating improves predictions against later observations.

Next, [L5b](../L5b/CHEME-5660-L5b-Lecture-MultipleAsset-GBM-Fall-2026.ipynb) extends the price model to several assets whose fluctuations can move together.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.
